In [4]:
# =============================================================================
# SETUP
# =============================================================================

# Uncomment if needed
# %pip install -U openai pandas tqdm

import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# =============================================================================
# CONFIGURATION
# =============================================================================

from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

MODEL = "gpt-4o"
# MODEL = "gpt-4.1-mini"   # cheaper alternative

INPUT_CSV = "data/solar_articles.csv"

OUTPUT_JSON = "graph/solar_graphs.json"
OUTPUT_NODES = "graph/nodes.csv"
OUTPUT_EDGES = "graph/edges.csv"

SAVE_CHECKPOINT_EVERY = 25

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(df)} articles")

# Optional:
# df = df.head(100)

Loaded 5374 articles


In [ ]:
# =============================================================================
# COUNT TOKENS IN ARTICLES
# =============================================================================

# Install if needed:
# %pip install tiktoken

import tiktoken
import pandas as pd

# Use the tokenizer closest to your model
encoding = tiktoken.encoding_for_model(MODEL)


def count_tokens(text):
    """
    Count tokens in a string using the model tokenizer.
    """
    if pd.isna(text):
        return 0
    
    return len(
        encoding.encode(str(text))
    )


# Count tokens in article text
df["token_count"] = df["text"].apply(count_tokens)

# Save updated CSV
df.to_csv(
    "data/solar_articles_w_tokens.csv",
    index=False
)


print(df[[
    "title",
    "token_count"
]].head())


print("\nToken statistics:")
print(df["token_count"].describe())


In [ ]:
"""
# Optional: count the full prompt size
def count_prompt_tokens(row):
    
    prompt = USER_PROMPT.format(
        article_id=row.name,
        publication_date=row["date"],
        source=row["archive"],
        title=row["title"],
        summary=row.get("summary", ""),
        text=row["text"],
    )
    
    return len(
        encoding.encode(
            SYSTEM_PROMPT + prompt
        )
    )


df["total_prompt_tokens"] = df.apply(
    count_prompt_tokens,
    axis=1
)
"""

In [ ]:
# =============================================================================
# SYSTEM PROMPT
# =============================================================================

SYSTEM_PROMPT = """
FILL IN
"""

# =============================================================================
# USER PROMPT TEMPLATE
# =============================================================================

USER_PROMPT = """
Article ID:
{article_id}

Publication Date:
{publication_date}

Source:
{source}

Title:
{title}

Summary:
{summary}

Article:

{text}
"""

# =============================================================================
# JSON SCHEMA
# =============================================================================

SOLAR_GRAPH_SCHEMA = {
    # FILL IN
}

# =============================================================================
# EXTRACTION FUNCTION
# =============================================================================

def extract_article(row):

    prompt = USER_PROMPT.format(
        article_id=row.name,
        publication_date=row["date"],
        source=row["archive"],
        title=row["title"],
        summary=row.get("summary", ""),
        text=row["text"],
    )

    response = client.chat.completions.create(

        model=MODEL,

        temperature=0,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        response_format={
            "type": "json_schema",
            "json_schema": SOLAR_GRAPH_SCHEMA
        }

    )

    return json.loads(response.choices[0].message.content)

# =============================================================================
# RUN EXTRACTION
# =============================================================================

results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    success = False

    for attempt in range(3):

        try:

            result = extract_article(row)

            results.append(result)

            success = True

            break

        except Exception as e:

            print(f"Row {idx} failed (attempt {attempt+1}): {e}")

            time.sleep(2)

    if not success:

        print(f"Skipping article {idx}")

    if len(results) % SAVE_CHECKPOINT_EVERY == 0:

        with open(OUTPUT_JSON, "w") as f:
            json.dump(results, f, indent=2)

# =============================================================================
# SAVE FINAL JSON
# =============================================================================

with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} extracted graphs.")

# =============================================================================
# FLATTEN NODES
# =============================================================================

nodes = []

for article in results:

    for node in article["nodes"]:

        node_copy = node.copy()

        node_copy["article_id"] = article["article_id"]

        nodes.append(node_copy)

nodes_df = pd.DataFrame(nodes)

nodes_df.to_csv(OUTPUT_NODES, index=False)

print(f"Saved {len(nodes_df)} nodes.")

# =============================================================================
# FLATTEN EDGES
# =============================================================================

edges = []

for article in results:

    edges.extend(article["edges"])

edges_df = pd.DataFrame(edges)

edges_df.to_csv(OUTPUT_EDGES, index=False)

print(f"Saved {len(edges_df)} edges.")

# =============================================================================
# OPTIONAL ANALYSIS
# =============================================================================

print(nodes_df.head())

print(edges_df.head())

print(edges_df["relationship"].value_counts())

print(nodes_df["type"].value_counts())

# Example:
#
# company_nodes = nodes_df[nodes_df["type"] == "Company"]
#
# predictions = edges_df[
#     edges_df["relationship"] == "PREDICTS"
# ]
#
# partnerships = edges_df[
#     edges_df["relationship"] == "PARTNERS_WITH"
# ]
#
# unrelated_articles = [
#     r for r in results if r["unrelated"]
# ]
#
# print(len(unrelated_articles))
